In [0]:
# Applies standardized naming rules across all Silver datasets.
# Uses the centrally defined SQL UDF (Python-based) to ensure consistent logic.
# This prevents schema drift and makes downstream Gold modeling easier.

# silver_utils
# Common reusable utilities for Silver layer

def standardize_columns(df, udf_fqn="coffee.silver.standardize_column_name"):
    """
    Standardizes DataFrame column names using the central SQL UDF.
    """
    rename_map = {}

    for col_name in df.columns:
        new_name = (
            spark.sql(f"SELECT {udf_fqn}('{col_name}') AS c")
            .collect()[0]["c"]
        )
        rename_map[col_name] = new_name

    df_std = df

    for old, new in rename_map.items():
        if old != new:
            df_std = df_std.withColumnRenamed(old, new)

    return df_std
